# Grocery Delivery Analytics Pipeline - 01 Bronze Generation

**Course:** SYST52461 - Big Data Storage and Analysis  
**Team:** Hazim Ali, Mannan, Maheshwar, Sweta, Omar Leopoldo, Shreyansh Pankaj

This notebook creates the dedicated Databricks catalog/schema and six deterministic Bronze Delta tables. Bronze preserves deliberately imperfect source values so the Silver notebook can demonstrate explicit data-quality decisions.

Run every cell from top to bottom. Re-running is safe because writes use overwrite mode.

## 1. Catalog, schema, imports, and shared helpers

In [ ]:
import random
from collections import defaultdict
from datetime import datetime, timedelta

from pyspark.sql import functions as F

CATALOG = "syst52461_grocery_delivery"
SCHEMA = "analytics"
SEED = 52461

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

def write_bronze(rows, table_name):
    df = spark.createDataFrame(rows)
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}_bronze"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(full_name))
    print(f"{full_name}: {df.count():,} rows")
    return df

def dirty_choice(value, variants, idx, every):
    return variants[(idx // every) % len(variants)] if idx % every == 0 else value

rng = random.Random(SEED)
print(f"Ready: {CATALOG}.{SCHEMA}; deterministic seed={SEED}")

## 2. Customers - owner: Hazim Ali

Quality issues: duplicate primary keys, null/uppercase emails, inconsistent city and loyalty casing, two date formats, missing ages, and an out-of-range age.

In [ ]:
cities = ["Toronto", "Mississauga", "Brampton", "Oakville", "Hamilton"]
loyalty_levels = ["Basic", "Silver", "Gold", "Platinum"]
customer_rows, customer_meta = [], {}

for customer_id in range(1, 1001):
    loyalty = rng.choices(loyalty_levels, weights=[0.46, 0.30, 0.17, 0.07], k=1)[0]
    city = rng.choice(cities)
    registered = datetime(2022, 1, 1) + timedelta(days=rng.randint(0, 1095))
    age = rng.randint(18, 74)
    customer_rows.append({
        "CustomerID": customer_id,
        "CustomerFullName": f"Customer {customer_id}",
        "Email": None if customer_id % 53 == 0 else (
            f"CUSTOMER{customer_id}@EXAMPLE.COM" if customer_id % 31 == 0 else f"customer{customer_id}@example.com"
        ),
        "City": dirty_choice(city, [city.lower(), city.upper(), f" {city} "], customer_id, 23),
        "LoyaltyStatus": dirty_choice(loyalty, [loyalty.lower(), loyalty.upper(), f" {loyalty} "], customer_id, 17),
        "RegistrationDateRaw": registered.strftime("%m/%d/%Y") if customer_id % 29 == 0 else registered.strftime("%Y-%m-%d"),
        "AgeRaw": None if customer_id % 41 == 0 else ("150" if customer_id % 137 == 0 else str(age)),
    })
    customer_meta[customer_id] = {"loyalty": loyalty, "city": city, "age": age}

for duplicate_id in [101, 202, 303, 404, 505]:
    customer_rows.append(dict(customer_rows[duplicate_id - 1]))

customers_bronze = write_bronze(customer_rows, "customers")
display(customers_bronze.limit(10))

## 3. Stores - owner: Mannan

Quality issues: inconsistent city/store-type casing, out-of-range rating, and missing rating.

In [ ]:
store_types = ["Supermarket", "Neighbourhood", "Express"]
store_rows, store_meta = [], {}

for store_id in range(1, 26):
    city = cities[(store_id - 1) % len(cities)]
    store_type = store_types[(store_id - 1) % len(store_types)]
    rating = round(3.5 + (store_id % 5) * 0.25 + rng.uniform(-0.15, 0.15), 1)
    rating_raw = "6.4" if store_id == 7 else (None if store_id == 19 else str(rating))
    store_rows.append({
        "StoreID": store_id,
        "StoreName": f"FreshRoute {city} {1 + (store_id - 1) // 5}",
        "City": city.lower() if store_id % 6 == 0 else city,
        "StoreType": store_type.upper() if store_id % 7 == 0 else store_type,
        "RatingRaw": rating_raw,
    })
    store_meta[store_id] = {
        "demand_weight": 0.75 + rating / 5,
        "delay_factor": (store_id % 5) * 1.7,
    }

stores_bronze = write_bronze(store_rows, "stores")
display(stores_bronze)

## 4. Products - owner: Maheshwar

Each store owns ten products. Quality issues: currency symbols, negative prices, missing costs/categories, inconsistent category casing, and negative inventory.

In [ ]:
category_ranges = {
    "Produce": (2.0, 12.0), "Dairy": (3.0, 15.0), "Bakery": (2.0, 11.0),
    "Meat & Seafood": (8.0, 34.0), "Pantry": (3.0, 22.0), "Frozen": (4.0, 19.0),
}
categories = list(category_ranges)
product_rows, product_meta, products_by_store = [], {}, defaultdict(list)
product_id = 1

for store_id in range(1, 26):
    for local_index in range(10):
        category = categories[(product_id + store_id) % len(categories)]
        low, high = category_ranges[category]
        price = round(rng.uniform(low, high), 2)
        cost = round(price * rng.uniform(0.56, 0.74), 2)
        price_raw = f"${price:.2f}" if product_id % 19 == 0 else f"{price:.2f}"
        if product_id % 61 == 0:
            price_raw = "-5.00"
        product_rows.append({
            "ProductID": product_id,
            "StoreID": store_id,
            "ProductName": f"{category} Item {local_index + 1}",
            "Category": None if product_id % 47 == 0 else dirty_choice(
                category, [category.lower(), category.upper(), f" {category} "], product_id, 13
            ),
            "UnitPriceRaw": price_raw,
            "UnitCostRaw": None if product_id % 83 == 0 else f"{cost:.2f}",
            "InventoryQtyRaw": "-4" if product_id % 89 == 0 else str(rng.randint(20, 450)),
        })
        product_meta[product_id] = {"store_id": store_id, "price": price, "cost": cost}
        products_by_store[store_id].append(product_id)
        product_id += 1

products_bronze = write_bronze(product_rows, "products")
display(products_bronze.limit(10))

## 5. Orders - owner: Sweta

Order selection intentionally gives loyalty members higher purchase frequency, weekends higher demand, and later months moderate growth. Quality issues: invalid foreign keys, inconsistent status/payment labels, and two timestamp formats.

In [ ]:
start, end = datetime(2025, 1, 1), datetime(2026, 6, 30)
dates, date_weights = [], []
cursor = start
total_days = (end - start).days + 1
while cursor <= end:
    progress = (cursor - start).days / total_days
    weight = (0.75 + progress * 0.65) * (1.32 if cursor.weekday() >= 5 else 1.0)
    if cursor.month == 12:
        weight *= 1.22
    dates.append(cursor); date_weights.append(weight); cursor += timedelta(days=1)

customer_ids = list(customer_meta)
loyalty_order_weight = {"Basic": 0.65, "Silver": 1.05, "Gold": 1.85, "Platinum": 2.70}
customer_weights = [loyalty_order_weight[customer_meta[c]["loyalty"]] for c in customer_ids]
store_ids = list(store_meta)
store_weights = [store_meta[s]["demand_weight"] for s in store_ids]

order_rows, order_meta = [], {}
for order_id in range(1, 4001):
    customer_id = rng.choices(customer_ids, weights=customer_weights, k=1)[0]
    store_id = rng.choices(store_ids, weights=store_weights, k=1)[0]
    order_day = rng.choices(dates, weights=date_weights, k=1)[0]
    order_time = order_day + timedelta(hours=rng.randint(7, 21), minutes=rng.randint(0, 59))
    status = rng.choices(["Completed", "Cancelled", "Pending"], weights=[91, 5, 4], k=1)[0]
    payment = rng.choices(["Credit Card", "Debit Card", "Digital Wallet", "Cash"], weights=[43, 26, 23, 8], k=1)[0]
    raw_customer_id = 999999 if order_id % 509 == 0 else customer_id
    raw_store_id = 9999 if order_id % 631 == 0 else store_id
    order_rows.append({
        "OrderID": order_id, "CustomerID": raw_customer_id, "StoreID": raw_store_id,
        "OrderTimestampRaw": order_time.strftime("%m/%d/%Y %H:%M") if order_id % 37 == 0 else order_time.strftime("%Y-%m-%d %H:%M:%S"),
        "OrderStatus": dirty_choice(status, [status.lower(), status.upper(), f" {status} "], order_id, 21),
        "PaymentMethod": dirty_choice(payment, [payment.lower(), payment.upper(), f" {payment} "], order_id, 33),
    })
    order_meta[order_id] = {"customer_id": customer_id, "store_id": store_id, "status": status}

orders_bronze = write_bronze(order_rows, "orders")
display(orders_bronze.limit(10))

## 6. Order items - owner: Omar Leopoldo

Quality issues: zero quantities, nonnumeric prices, currency symbols, discount percentages represented as `0.15`, `15`, or `15%`, and rare store/product mismatches.

In [ ]:
item_rows = []
item_id = 1
for order_id, order in order_meta.items():
    loyalty = customer_meta[order["customer_id"]]["loyalty"]
    item_count = rng.choices([1, 2, 3, 4, 5], weights=[12, 28, 31, 20, 9], k=1)[0]
    if loyalty in {"Gold", "Platinum"} and rng.random() < 0.32:
        item_count = min(6, item_count + 1)
    for _ in range(item_count):
        store_id = order["store_id"]
        product_id = rng.choice(products_by_store[store_id])
        if item_id % 997 == 0:
            product_id = rng.choice(products_by_store[(store_id % 25) + 1])
        discount = rng.choices([0.0, 0.05, 0.10, 0.15, 0.20], weights=[37, 21, 20, 14, 8], k=1)[0]
        qty_weights = [62, 27, 9, 2] if discount < 0.15 else [39, 34, 20, 7]
        quantity = rng.choices([1, 2, 3, 4], weights=qty_weights, k=1)[0]
        if item_id % 173 == 0:
            quantity = 0
        price = product_meta[product_id]["price"]
        price_raw = f"${price:.2f}" if item_id % 29 == 0 else f"{price:.2f}"
        if item_id % 211 == 0:
            price_raw = "not available"
        discount_raw = f"{discount * 100:.0f}%" if item_id % 31 == 0 else (
            f"{discount * 100:.0f}" if item_id % 17 == 0 else f"{discount:.2f}"
        )
        item_rows.append({
            "OrderItemID": item_id, "OrderID": order_id, "ProductID": product_id,
            "QuantityRaw": str(quantity), "UnitPriceRaw": price_raw, "DiscountRaw": discount_raw,
        })
        item_id += 1

order_items_bronze = write_bronze(item_rows, "order_items")
display(order_items_bronze.limit(10))

## 7. Deliveries - owner: Shreyansh Pankaj

Quality issues: units embedded in numeric strings, negative distances, inconsistent delivery status, and missing actual times for unfinished deliveries. Distance and store operational factors drive realistic delays.

In [ ]:
delivery_rows = []
for order_id, order in order_meta.items():
    distance = round(rng.uniform(0.8, 24.0), 1)
    promised = round(24 + 1.55 * distance + rng.uniform(-2, 3), 1)
    actual = round(17 + 2.25 * distance + store_meta[order["store_id"]]["delay_factor"] + rng.gauss(0, 5.5), 1)
    actual = max(actual, 9.0)
    delivery_status = "Delivered" if order["status"] == "Completed" else (
        "Cancelled" if order["status"] == "Cancelled" else "In Transit"
    )
    distance_raw = f"{distance} km" if order_id % 27 == 0 else f"{distance}"
    if order_id % 521 == 0:
        distance_raw = "-3.0"
    delivery_rows.append({
        "DeliveryID": order_id, "OrderID": order_id,
        "DriverID": 1000 + rng.randint(1, 180), "DriverName": f"Driver {1 + order_id % 180}",
        "DeliveryStatus": dirty_choice(delivery_status, [delivery_status.lower(), delivery_status.upper(), f" {delivery_status} "], order_id, 25),
        "DistanceKmRaw": distance_raw,
        "PromisedMinutesRaw": f"{promised} min" if order_id % 43 == 0 else f"{promised}",
        "ActualMinutesRaw": None if delivery_status != "Delivered" else (
            f"{actual} min" if order_id % 47 == 0 else f"{actual}"
        ),
    })

deliveries_bronze = write_bronze(delivery_rows, "deliveries")
display(deliveries_bronze.limit(10))

## 8. Bronze inventory and quality snapshot

In [ ]:
bronze_tables = ["customers", "stores", "products", "orders", "order_items", "deliveries"]
inventory = []
for name in bronze_tables:
    df = spark.table(f"{CATALOG}.{SCHEMA}.{name}_bronze")
    null_total = sum(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first())
    inventory.append((name, df.count(), len(df.columns), int(null_total)))

inventory_df = spark.createDataFrame(inventory, ["table_name", "row_count", "column_count", "null_value_count"])
display(inventory_df.orderBy("table_name"))

assert customers_bronze.count() == 1005
assert stores_bronze.count() == 25
assert products_bronze.count() == 250
assert orders_bronze.count() == 4000
assert deliveries_bronze.count() == 4000
print("Bronze generation complete. Continue with 02_silver_processing.ipynb.")